In [1]:
import torch

# Force GPU 1 (since GPU 0 has ECC error)
DEVICE = torch.device("cuda:1")
torch.cuda.set_device(1)

print("Using device:", torch.cuda.get_device_name(1))

Using device: Tesla V100-PCIE-16GB


In [2]:
print("Current device index:", torch.cuda.current_device())

Current device index: 1


In [4]:
# ============================================================
# CELL 1 — CONFIGURATION
# Target: 98%+ accuracy | 10 Runs | Full architecture alignment
# Architecture: ConvNeXt + MaxViT | SE Attention | Hybrid Loss
# Ensemble: 5-Fold × 3 Seeds = 15 Models per Run × 10 Runs
# ============================================================

# ------------------------------------------------------------
# FORCE CLEAN GPU (Avoid GPU 0 with ECC error)
# ------------------------------------------------------------
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # ← USE GPU 1 (GPU 0 has ECC error)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
import subprocess, sys, gc, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from PIL import Image
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import csv
from datetime import datetime
from IPython.display import display

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True

# ------------------------------------------------------------
# Core Settings
# ------------------------------------------------------------
TEST_MODE   = False
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE    = 224
BATCH_SIZE  = 4
NUM_CLASSES = 5

# ------------------------------------------------------------
# Training Depth (Architecture Aligned)
# ------------------------------------------------------------
NUM_EPOCHS  = 40
NUM_FOLDS   = 5
NUM_SEEDS   = 3
NUM_RUNS    = 10

# ------------------------------------------------------------
# Hybrid Loss Weights
# ------------------------------------------------------------
ALPHA = 0.5   # CrossEntropy
BETA  = 0.2   # MSE
GAMMA = 0.3   # QWK (ordinal penalty)

# ------------------------------------------------------------
# Memory & Speed
# ------------------------------------------------------------
GRAD_ACCUM  = 8     # effective batch = 32
AMP         = True
NUM_WORKERS = 0
PIN_MEMORY  = torch.cuda.is_available()

# ------------------------------------------------------------
# Learning Rate
# ------------------------------------------------------------
LR = 2e-4
MAX_SAMPLES = None

# ------------------------------------------------------------
# Paths (Server Paths Example)
# ------------------------------------------------------------
DATASET_PATHS = {
    "Messidor2": "/home/user2007/Messsidor-2"
}

OUTPUT_DIR = "/home/user2007/Dr_retinopathy/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Configuration Summary
# ------------------------------------------------------------
print("=" * 60)
print("  CONFIGURATION SUMMARY")
print("=" * 60)
print(f"  Device      : {DEVICE}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free  = props.total_memory - torch.cuda.memory_allocated(0)

    print(f"  GPU         : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM Total  : {props.total_memory/1e9:.1f} GB")
    print(f"  VRAM Free   : {free/1e9:.1f} GB")

print(f"  IMG_SIZE    : {IMG_SIZE}")
print(f"  BATCH_SIZE  : {BATCH_SIZE}  (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"  NUM_EPOCHS  : {NUM_EPOCHS}")
print(f"  NUM_FOLDS   : {NUM_FOLDS}  (5-Fold CV)")
print(f"  NUM_SEEDS   : {NUM_SEEDS}  (3 seeds/fold = 15 models/run)")
print(f"  NUM_RUNS    : {NUM_RUNS}  (metrics averaged over runs)")
print(f"  Loss α/β/γ  : {ALPHA}/{BETA}/{GAMMA}  (CE/MSE/QWK)")
print(f"  AMP         : {AMP}")
print(f"  LR          : {LR}")
print("=" * 60)

# ------------------------------------------------------------
# Time Estimation
# ------------------------------------------------------------
imgs_train    = 1748 * 0.84
steps         = imgs_train / (BATCH_SIZE * GRAD_ACCUM)
secs_epoch    = steps * 0.8
models_total  = NUM_FOLDS * NUM_SEEDS * NUM_RUNS
total_secs    = secs_epoch * NUM_EPOCHS * models_total
total_hours   = total_secs / 3600

print(f"\n  ⏱  Estimated time  : {total_hours:.0f} – {total_hours*1.3:.0f} hours")
print(f"  🧠  Total models   : {models_total} "
      f"({NUM_FOLDS} folds × {NUM_SEEDS} seeds × {NUM_RUNS} runs)")
print(f"  📁  Output dir     : {OUTPUT_DIR}")

print("\n  💡  TIME SAVER TIP:")
print("      Set NUM_RUNS = 1 first to validate pipeline.")
print("      Then launch full 10-run experiment overnight.")

print("\n✅ CELL 1 complete")

/home/user2007/Dr_retinopathy/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  CONFIGURATION SUMMARY
  Device      : cuda
  GPU         : Tesla V100-PCIE-16GB
  VRAM Total  : 16.9 GB
  VRAM Free   : 16.9 GB
  IMG_SIZE    : 224
  BATCH_SIZE  : 4  (effective: 32)
  NUM_EPOCHS  : 40
  NUM_FOLDS   : 5  (5-Fold CV)
  NUM_SEEDS   : 3  (3 seeds/fold = 15 models/run)
  NUM_RUNS    : 10  (metrics averaged over runs)
  Loss α/β/γ  : 0.5/0.2/0.3  (CE/MSE/QWK)
  AMP         : True
  LR          : 0.0002

  ⏱  Estimated time  : 61 – 80 hours
  🧠  Total models   : 150 (5 folds × 3 seeds × 10 runs)
  📁  Output dir     : /home/user2007/Dr_retinopathy/results

  💡  TIME SAVER TIP:
      Set NUM_RUNS = 1 first to validate pipeline.
      Then launch full 10-run experiment overnight.

✅ CELL 1 complete


In [5]:
# ============================================================
# CELL 2 — DATASET LOADER (Messidor-2) — FIXED VERSION
# ============================================================

import os
import pandas as pd
from sklearn.model_selection import train_test_split

# ---- IMPORTANT: SET CORRECT PATH HERE ----
DATASET_PATHS = {
    "Messidor2": "/home/user2007/Messsidor-2"   # <-- 3 s
}

NUM_CLASSES = 5
MAX_SAMPLES = None   # set number if you want cap

MESSIDOR_ROOT = DATASET_PATHS["Messidor2"]
csv_path = os.path.join(MESSIDOR_ROOT, "messidor_data.csv")

print("=" * 60)
print("DIAGNOSING messidor_data.csv")
print("=" * 60)

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV not found at {csv_path}")

df_raw = pd.read_csv(csv_path)
df_raw.columns = [c.strip() for c in df_raw.columns]

print(f"Shape       : {df_raw.shape}")
print(f"Columns     : {list(df_raw.columns)}")
print(df_raw.head(3))


# ------------------------------------------------------------
# AUTO COLUMN DETECTION
# ------------------------------------------------------------

def find_column(columns, keywords):
    for kw in keywords:
        for c in columns:
            if kw in c.lower():
                return c
    return None


id_col = find_column(df_raw.columns, 
                     ["image", "file", "name", "id", "img", "path"]) or df_raw.columns[0]

label_col = find_column(df_raw.columns, 
                        ["grade", "label", "class", "diag", "severity", "level"]) or df_raw.columns[-1]

print(f"✔ ID_COL    = {id_col}")
print(f"✔ LABEL_COL = {label_col}")


# ------------------------------------------------------------
# CLEAN DATAFRAME
# ------------------------------------------------------------

df = df_raw.rename(columns={id_col: "image_id", label_col: "label"})

df["label"] = pd.to_numeric(df["label"], errors="coerce")
df = df.dropna(subset=["label"])
df["label"] = df["label"].astype(int)

df = df[df["label"].between(0, NUM_CLASSES - 1)].reset_index(drop=True)


# ------------------------------------------------------------
# FIND IMAGE DIRECTORY (SIMPLIFIED & FIXED)
# ------------------------------------------------------------

possible_dirs = [
    os.path.join(MESSIDOR_ROOT, "preprocess"),
    os.path.join(MESSIDOR_ROOT, "images"),
    os.path.join(MESSIDOR_ROOT, "messidor-2"),
    MESSIDOR_ROOT
]

img_dir = None

for d in possible_dirs:
    if os.path.isdir(d):
        files = [f for f in os.listdir(d) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        if files:
            img_dir = d
            print(f"✔ Image folder found: {img_dir}")
            print(f"Sample files: {files[:3]}")
            break



# ------------------------------------------------------------
# FIXED IMAGE DIRECTORY (BASED ON YOUR SERVER STRUCTURE)
# ------------------------------------------------------------

img_dir = os.path.join(
    MESSIDOR_ROOT,
    "messidor-2",
    "messidor-2",
    "preprocess"
)

if not os.path.isdir(img_dir):
    raise FileNotFoundError(f"Image folder not found at {img_dir}")

print(f"✔ Image folder found: {img_dir}")
print("Sample files:", os.listdir(img_dir)[:3])
# ------------------------------------------------------------
# MATCH IMAGE PATHS
# ------------------------------------------------------------

def find_image(img_id):
    img_id = str(img_id).strip()
    base = os.path.splitext(img_id)[0]

    for ext in ["", ".png", ".jpg", ".jpeg", ".PNG", ".JPG"]:
        path = os.path.join(img_dir, img_id + ext)
        if os.path.exists(path):
            return path

        path = os.path.join(img_dir, base + ext)
        if os.path.exists(path):
            return path

    return None


df["path"] = df["image_id"].apply(find_image)
df = df[df["path"].notnull()].reset_index(drop=True)

print(f"✔ Valid images: {len(df)}")
print("Class Distribution:", df["label"].value_counts().sort_index().to_dict())


# ------------------------------------------------------------
# TRAIN / VAL / TEST SPLIT
# ------------------------------------------------------------

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print(f"✔ Split → Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}")

print("\n✅ Messidor-2 loader ready!")

DIAGNOSING messidor_data.csv
Shape       : (1744, 4)
Columns     : ['id_code', 'diagnosis', 'adjudicated_dme', 'adjudicated_gradable']
                      id_code  diagnosis  adjudicated_dme  \
0  20051020_43808_0100_PP.png          0                0   
1  20051020_43832_0100_PP.png          1                0   
2  20051020_43882_0100_PP.png          1                0   

   adjudicated_gradable  
0                     1  
1                     1  
2                     1  
✔ ID_COL    = id_code
✔ LABEL_COL = diagnosis
✔ Image folder found: /home/user2007/Messsidor-2/messidor-2/messidor-2/preprocess
Sample files: ['IM003390.JPG', '20051020_44901_0100_PP.png', '20051117_37185_0400_PP.png']
✔ Valid images: 1744
Class Distribution: {0: 1017, 1: 270, 2: 347, 3: 75, 4: 35}
✔ Split → Train=1220  Val=262  Test=262

✅ Messidor-2 loader ready!


In [6]:
# ============================================================
# CELL 3 — DATASET DISTRIBUTION (displayed inline)
# ============================================================

CLASS_NAMES = ["Grade 0\n(No DR)", "Grade 1\n(Mild DR)", "Grade 2\n(Moderate DR)",
               "Grade 3\n(Severe DR)", "Grade 4\n(PDR)"]
COLORS = ["#4CAF50", "#FFC107", "#FF9800", "#F44336", "#9C27B0"]

def plot_distribution(dataset_name, train_df, val_df, test_df):
    splits = {"Training": train_df, "Validation": val_df, "Testing": test_df}
    fig, axes = plt.subplots(1, 4, figsize=(24, 7))
    fig.suptitle(f"📊  {dataset_name} — Dataset Distribution",
                 fontsize=15, fontweight="bold", y=1.02)

    all_df = pd.concat([train_df, val_df, test_df])
    all_counts = all_df["label"].value_counts().sort_index()
    axes[0].pie(all_counts.values,
                labels=[CLASS_NAMES[i] for i in all_counts.index],
                colors=[COLORS[i] for i in all_counts.index],
                autopct="%1.1f%%", startangle=140,
                textprops={"fontsize": 7.5},
                wedgeprops={"edgecolor": "white", "linewidth": 1.2})
    axes[0].set_title(f"Overall\n(n={len(all_df):,})", fontweight="bold")

    for ax, (split_name, df) in zip(axes[1:], splits.items()):
        counts = df["label"].value_counts().sort_index()
        ax.pie(counts.values,
               labels=[CLASS_NAMES[i] for i in counts.index],
               colors=[COLORS[i] for i in counts.index],
               autopct="%1.1f%%", startangle=140,
               textprops={"fontsize": 7.5},
               wedgeprops={"edgecolor": "white", "linewidth": 1.2})
        ax.set_title(f"{split_name}\n(n={len(df):,})", fontweight="bold")

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_distribution.png")
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()   # ← inline display

    print(f"\n  {'Grade':<10} {'Train':>8} {'Val':>8} {'Test':>8}")
    print(f"  {'-'*38}")
    for g in range(NUM_CLASSES):
        print(f"  Grade {g:<4}  {(train_df['label']==g).sum():>8} "
              f"{(val_df['label']==g).sum():>8} "
              f"{(test_df['label']==g).sum():>8}")
    print(f"  📁 Saved → {save_path}")

print("✅ CELL 3 complete")


✅ CELL 3 complete


In [7]:
# ============================================================
# CELL 4 — PREPROCESSING VISUALISATION (displayed inline)
# ============================================================

def _apply_clahe(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR)

def _apply_normalization(img_bgr):
    img_f = img_bgr.astype(np.float32) / 255.0
    mean  = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std   = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img_norm = (img_f - mean) / std
    return (np.clip(img_norm * std + mean, 0, 1) * 255).astype(np.uint8)

def _apply_augmentation(img_bgr):
    aug = A.Compose([
        A.HorizontalFlip(p=1.0),
        A.RandomRotate90(p=1.0),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1, p=1.0),
    ])
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return cv2.cvtColor(aug(image=rgb)["image"], cv2.COLOR_RGB2BGR)

def visualise_preprocessing(sample_path, dataset_name):
    raw = cv2.imread(sample_path)
    if raw is None:
        print(f"  ⚠️  Cannot read: {sample_path}"); return
    raw       = cv2.resize(raw, (IMG_SIZE, IMG_SIZE))
    clahe_img = _apply_clahe(raw)
    norm_img  = _apply_normalization(clahe_img)
    aug_img   = _apply_augmentation(norm_img)

    steps = [
        ("Original\nFundus Image",   cv2.cvtColor(raw,       cv2.COLOR_BGR2RGB)),
        ("After CLAHE\nEnhancement", cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB)),
        ("After\nNormalization",      cv2.cvtColor(norm_img,  cv2.COLOR_BGR2RGB)),
        ("After\nAugmentation",       cv2.cvtColor(aug_img,   cv2.COLOR_BGR2RGB)),
    ]
    border_colors = ["#607D8B", "#2196F3", "#4CAF50", "#FF9800"]

    fig, axes = plt.subplots(1, 4, figsize=(20, 6))
    fig.suptitle(f"🔬  {dataset_name} — Preprocessing Pipeline",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, (title, img), bc in zip(axes, steps, border_colors):
        ax.imshow(img)
        ax.set_title(title, fontweight="bold", fontsize=11, pad=8)
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_edgecolor(bc); spine.set_linewidth(3); spine.set_visible(True)

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_preprocessing.png")
    plt.savefig(save_path, dpi=120, bbox_inches="tight")
    plt.show()   # ← inline display
    print(f"  📁 Saved → {save_path}")

print("✅ CELL 4 complete")


✅ CELL 4 complete


In [8]:
# ============================================================
# CELL 5 — DATASET CLASS & AUGMENTATION
# Architecture: CLAHE → Normalization → Augmentation pipeline
# ============================================================

def get_transforms(phase):
    if phase == "train":
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            # Geometric
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15,
                               rotate_limit=45, border_mode=cv2.BORDER_REFLECT, p=0.7),
            A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3),
            A.ElasticTransform(alpha=1, sigma=10, p=0.2),
            # Color / photometric
            A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.8),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.CLAHE(clip_limit=3.0, tile_grid_size=(8, 8), p=0.4),
            # Noise & blur
            A.GaussNoise(var_limit=(10, 60), p=0.4),
            A.GaussianBlur(blur_limit=3, p=0.3),
            A.MotionBlur(blur_limit=3, p=0.2),
            # Dropout
            A.CoarseDropout(max_holes=8, max_height=20, max_width=20, fill_value=0, p=0.3),
            # Normalize (ImageNet stats — aligns with backbone pretraining)
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    elif phase == "tta":
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])
    else:   # val / test — clean pass
        return A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ])


class DRDataset(Dataset):
    """
    Architecture pipeline:
    Image → CLAHE Enhancement → Normalization → Augmentation → Tensor
    """
    def __init__(self, df, phase="train"):
        self.df        = df.reset_index(drop=True)
        self.phase     = phase
        self.transform = get_transforms(phase)
        self._clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

    def __len__(self): return len(self.df)

    def _load(self, path):
        img = cv2.imread(str(path))
        if img is None:
            return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        # CLAHE Enhancement (architecture step 1)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l = self._clahe.apply(l)
        img = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row["label"]) if row["label"] >= 0 else 0
        img   = self._load(row["path"])
        if self.transform:
            img = self.transform(image=img)["image"]
        return (img,
                torch.tensor(label, dtype=torch.long),
                torch.tensor(label, dtype=torch.float32))


def make_loader(df, phase, num_workers=NUM_WORKERS):
    ds = DRDataset(df, phase)
    if phase == "train":
        # Class-weighted sampler — handles class imbalance
        labels    = df["label"].values
        class_cnt = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
        class_cnt = np.where(class_cnt == 0, 1, class_cnt)
        weights   = 1.0 / class_cnt
        sample_w  = torch.tensor([weights[l] for l in labels], dtype=torch.float32)
        sampler   = torch.utils.data.WeightedRandomSampler(
            sample_w, num_samples=len(sample_w), replacement=True)
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=num_workers, pin_memory=PIN_MEMORY, drop_last=True)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=num_workers, pin_memory=PIN_MEMORY)

print("✅ CELL 5 complete")


✅ CELL 5 complete


In [9]:
# ============================================================
# CELL 6 — MODEL ARCHITECTURE
# ┌─────────────────────────────────────────────────────┐
# │  ConvNeXt-Tiny (Pretrained) ──┐                     │
# │                               ├→ Concat → SE → LN   │
# │  MaxViT-Tiny (Pretrained)  ───┘                     │
# │         ↓                ↓                          │
# │  Classification Head    Regression Head             │
# │  (FC→512→256→5 cls)    (FC→256→128→1 reg)          │
# └─────────────────────────────────────────────────────┘
# ============================================================

class SEBlock(nn.Module):
    """Squeeze-and-Excitation Attention Module (architecture diagram)"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x)

class DRModel(nn.Module):
    def __init__(self, num_classes=5, drop_rate=0.4):
        super().__init__()
        # ── Parallel Feature Extraction ─────────────────────
        self.convnext = timm.create_model("convnext_tiny", pretrained=True, num_classes=0)
        self.maxvit   = timm.create_model("maxvit_tiny_tf_224", pretrained=True, num_classes=0)
        feat1, feat2  = self.convnext.num_features, self.maxvit.num_features
        combined      = feat1 + feat2

        # ── Feature Fusion ───────────────────────────────────
        # Feature Concatenation → SE Attention → Combined Feature Vector
        self.se   = SEBlock(combined)
        self.norm = nn.LayerNorm(combined)

        # ── Multi-Task Learning Head ─────────────────────────
        # Classification Branch: FC → 5-Class Output → Softmax
        self.cls_head = nn.Sequential(
            nn.Dropout(drop_rate),
            nn.Linear(combined, 512), nn.GELU(),
            nn.Dropout(drop_rate / 2),
            nn.Linear(512, 256), nn.GELU(),
            nn.Linear(256, num_classes),
        )
        # Regression Branch: FC → Severity Score (0-4) → Linear
        self.reg_head = nn.Sequential(
            nn.Dropout(drop_rate),
            nn.Linear(combined, 256), nn.GELU(),
            nn.Dropout(drop_rate / 2),
            nn.Linear(256, 128), nn.GELU(),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        f1   = self.convnext(x)          # ConvNeXt features
        f2   = self.maxvit(x)            # MaxViT features
        feat = torch.cat([f1, f2], dim=1)  # Feature Concatenation
        feat = self.se(feat)             # SE Attention Module
        feat = self.norm(feat)           # Combined Feature Vector
        return self.cls_head(feat), self.reg_head(feat).squeeze(1)

def get_model():
    return DRModel(NUM_CLASSES).to(DEVICE)

_m = get_model()
total_params = sum(p.numel() for p in _m.parameters())
print(f"  Total parameters : {total_params:,}")
print(f"  ConvNeXt features: {_m.convnext.num_features}")
print(f"  MaxViT features  : {_m.maxvit.num_features}")
print(f"  Combined features: {_m.convnext.num_features + _m.maxvit.num_features}")
del _m; torch.cuda.empty_cache()
print("✅ CELL 6 complete")


  Total parameters : 59,580,462
  ConvNeXt features: 768
  MaxViT features  : 512
  Combined features: 1280
✅ CELL 6 complete


In [11]:
# ============================================================
# CELL 7 — HYBRID LOSS FUNCTION
# Loss = α·CE + β·MSE + γ·QWK
# α=0.5 (CrossEntropy), β=0.2 (MSE), γ=0.3 (QWK ordinal)
# ============================================================

def quadratic_weighted_kappa_loss(logits, targets, num_classes=NUM_CLASSES):
    """Ordinal Penalty — QWK-based differentiable loss"""
    probs      = F.softmax(logits, dim=1)
    levels     = torch.arange(num_classes, device=logits.device, dtype=torch.float32)
    pred_score = (probs * levels).sum(dim=1)
    target_f   = targets.float()
    num        = ((pred_score - target_f) ** 2).mean()
    denom      = (pred_score.var() + target_f.var()).clamp(min=1e-8)
    return num / denom


class HybridLoss(nn.Module):
    """
    Hybrid Loss (architecture diagram):
      α · CrossEntropy  (Classification Loss)
    + β · MSE           (Regression Loss)
    + γ · QWK           (Ordinal Penalty)
    """
    def __init__(self, alpha=ALPHA, beta=BETA, gamma=GAMMA):
        super().__init__()
        self.alpha = alpha; self.beta = beta; self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.mse   = nn.MSELoss()

    def forward(self, logits, reg_out, cls_labels, reg_labels):
        l_ce  = self.ce(logits, cls_labels)
        l_mse = self.mse(reg_out, reg_labels)
        l_qwk = quadratic_weighted_kappa_loss(logits, cls_labels)
        return self.alpha * l_ce + self.beta * l_mse + self.gamma * l_qwk

print("  Hybrid Loss: α·CE + β·MSE + γ·QWK")
print(f"  Weights   : α={ALPHA}, β={BETA}, γ={GAMMA}")
print("✅ CELL 7 complete")


  Hybrid Loss: α·CE + β·MSE + γ·QWK
  Weights   : α=0.5, β=0.2, γ=0.3
✅ CELL 7 complete


In [12]:
# ============================================================
# CELL 8 — TRAIN / EVAL / PREDICT WITH TTA
# TTA: 8 forward passes → average probabilities (more robust)
# ============================================================

_scaler = torch.cuda.amp.GradScaler(enabled=AMP)

def train_epoch(model, loader, optimizer, criterion, scheduler=None):
    model.train()
    total_loss = 0.0; correct = 0; total = 0
    optimizer.zero_grad()
    for step, (imgs, cls_lbl, reg_lbl) in enumerate(loader):
        imgs    = imgs.to(DEVICE, non_blocking=True)
        cls_lbl = cls_lbl.to(DEVICE, non_blocking=True)
        reg_lbl = reg_lbl.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=AMP):
            logits, reg_out = model(imgs)
            loss = criterion(logits, reg_out, cls_lbl, reg_lbl) / GRAD_ACCUM

        _scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(loader):
            _scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            _scaler.step(optimizer)
            _scaler.update()
            optimizer.zero_grad()
            if scheduler:
                scheduler.step()

        total_loss += loss.item() * GRAD_ACCUM
        correct    += (logits.argmax(1) == cls_lbl).sum().item()
        total      += cls_lbl.size(0)

    return total_loss / len(loader), correct / max(total, 1)


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0; all_preds = []; all_labels = []
    for imgs, cls_lbl, reg_lbl in loader:
        imgs    = imgs.to(DEVICE, non_blocking=True)
        cls_lbl = cls_lbl.to(DEVICE, non_blocking=True)
        reg_lbl = reg_lbl.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=AMP):
            logits, reg_out = model(imgs)
            loss = criterion(logits, reg_out, cls_lbl, reg_lbl)
        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(cls_lbl.cpu().numpy())
    return (total_loss / max(len(loader), 1),
            accuracy_score(all_labels, all_preds),
            np.array(all_preds), np.array(all_labels))


@torch.no_grad()
def predict(model, df, n_tta=8):
    """
    Ensemble Strategy (architecture): Average Logits → Softmax
    n_tta TTA passes (first pass clean, rest augmented)
    """
    model.eval()
    all_logits = []
    for tta_idx in range(n_tta):
        phase      = "tta" if tta_idx > 0 else "val"
        loader     = make_loader(df, phase)
        run_logits = []
        for imgs, _, __ in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=AMP):
                logits, _ = model(imgs)
            run_logits.append(logits.cpu())
        all_logits.append(torch.cat(run_logits, dim=0))

    # Average Logits → Softmax (architecture ensemble strategy)
    probs_stack = torch.stack([torch.softmax(l, dim=1) for l in all_logits])
    avg_probs   = probs_stack.mean(0)
    return torch.log(avg_probs + 1e-8)

print("✅ CELL 8 complete")


✅ CELL 8 complete


In [13]:
# ============================================================
# CELL 9 — METRICS, CSV LOGGING & VISUALISATION
# All metrics saved to CSV; averaged metrics shown as bar graph
# ============================================================

CSV_FIELDNAMES = ["dataset", "run", "fold", "seed",
                  "accuracy", "precision", "sensitivity", "specificity", "f1_score"]

def compute_metrics(y_true, y_pred, dataset_name, run_id, fold=None, seed=None):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred,    average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred,        average="weighted", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    specs = []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        specs.append(tn / (tn + fp + 1e-8))
    return {
        "dataset":     dataset_name,
        "run":         run_id,
        "fold":        fold if fold is not None else "all",
        "seed":        seed if seed is not None else "all",
        "accuracy":    round(float(acc),  4),
        "precision":   round(float(prec), 4),
        "sensitivity": round(float(rec),  4),
        "specificity": round(float(np.mean(specs)), 4),
        "f1_score":    round(float(f1),   4),
    }


def save_confusion_matrix(y_true, y_pred, dataset_name, run_id, tag=""):
    cm     = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    labels = ["Grade 0\n(No DR)", "Grade 1\n(Mild)", "Grade 2\n(Mod.)",
              "Grade 3\n(Severe)", "Grade 4\n(PDR)"]
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax,
                linewidths=0.5, linecolor="white", annot_kws={"fontsize": 11})
    ax.set_title(f"{dataset_name} — Confusion Matrix  Run {run_id} {tag}",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.tight_layout()
    plt.show()   # ← inline display
    fpath = os.path.join(OUTPUT_DIR, f"{dataset_name}_run{run_id}{tag}_confusion_matrix.png")
    plt.savefig(fpath, dpi=110, bbox_inches="tight")
    plt.close()
    print(f"    📁 Confusion matrix → {fpath}")


def append_metrics_csv(metrics_list, dataset_name):
    csv_path     = os.path.join(OUTPUT_DIR, f"{dataset_name}_metrics.csv")
    write_header = not os.path.exists(csv_path)
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES)
        if write_header:
            writer.writeheader()
        for m in metrics_list:
            writer.writerow({k: m.get(k, "") for k in CSV_FIELDNAMES})
    return csv_path


def plot_averaged_metrics_bar(all_metrics, dataset_name):
    """
    Compute averaged metrics across all runs and display as bar chart (inline).
    Also saves the bar chart and appends the AVERAGE row to the CSV.
    """
    df_m     = pd.DataFrame(all_metrics)
    num_cols = ["accuracy", "precision", "sensitivity", "specificity", "f1_score"]
    avg      = df_m[num_cols].mean().round(4)

    # ── Append AVERAGE row to CSV ────────────────────────────
    avg_row  = {"dataset": dataset_name, "run": "AVERAGE",
                "fold": "ALL", "seed": "ALL", **avg.to_dict()}
    csv_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_metrics.csv")
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDNAMES)
        writer.writerow(avg_row)

    # ── Bar chart of averaged metrics ────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    bar_colors = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0", "#F44336"]
    bars = ax.bar(num_cols, avg.values * 100, color=bar_colors,
                  edgecolor="white", linewidth=1.2, width=0.55)

    # Annotate bars
    for bar, val in zip(bars, avg.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f"{val*100:.2f}%",
                ha="center", va="bottom", fontweight="bold", fontsize=12)

    ax.set_ylim(80, 102)
    ax.set_yticks(range(80, 103, 2))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))
    ax.set_xlabel("Metric", fontsize=13, fontweight="bold")
    ax.set_ylabel("Score (%)", fontsize=13, fontweight="bold")
    ax.set_title(f"📊  {dataset_name} — Averaged Metrics ({NUM_RUNS} Runs × {NUM_FOLDS} Folds × {NUM_SEEDS} Seeds)",
                 fontsize=13, fontweight="bold", pad=14)
    ax.set_xticklabels([c.replace("_", "\n").title() for c in num_cols], fontsize=11)
    ax.axhline(y=98, color="red", linestyle="--", linewidth=1.5, label="98% target")
    ax.legend(fontsize=11)
    ax.grid(axis="y", alpha=0.3)
    sns.despine()

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, f"{dataset_name}_averaged_metrics_bar.png")
    plt.savefig(chart_path, dpi=120, bbox_inches="tight")
    plt.show()   # ← inline display
    print(f"  📁 Bar chart → {chart_path}")

    # ── Console table ─────────────────────────────────────────
    print(f"\n  ╔══════════════════════════════════════════════╗")
    print(f"  ║  {dataset_name:^44s}  ║")
    print(f"  ║  AVERAGED METRICS ({NUM_RUNS} runs)                  ║")
    print(f"  ╠══════════════════════════════════════════════╣")
    for k in num_cols:
        bar_len = int(avg[k] * 30)
        bar     = "█" * bar_len + "░" * (30 - bar_len)
        print(f"  ║  {k:<15s}: {avg[k]:.4f}  [{bar}] ║")
    print(f"  ╚══════════════════════════════════════════════╝")
    return avg

print("✅ CELL 9 complete")


✅ CELL 9 complete


In [14]:
# ============================================================
# CELL 10 — TRAINING LOOP
# Architecture: 5-Fold × 3 Seeds = 15 models → Average Logits
# Differential LR: backbone × 0.1, heads × 1.0
# Cosine Annealing with Warm Restarts
# Early stopping patience = 10 epochs
# ============================================================

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _train_one_model(fold_tr, fold_vl, test_df, seed, run_id, fold, criterion):
    set_seed(seed + run_id * 100 + fold * 10)
    model = get_model()

    # ── Differential Learning Rates ──────────────────────────
    backbone_params = (list(model.convnext.parameters()) +
                       list(model.maxvit.parameters()))
    head_params     = (list(model.se.parameters()) +
                       list(model.norm.parameters()) +
                       list(model.cls_head.parameters()) +
                       list(model.reg_head.parameters()))

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": LR * 0.1},  # backbone: 10× lower LR
        {"params": head_params,     "lr": LR},          # heads: full LR
    ], weight_decay=1e-2)

    tr_loader = make_loader(fold_tr, "train")
    vl_loader = make_loader(fold_vl, "val")

    # ── Cosine Annealing with Warm Restarts ──────────────────
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2, eta_min=1e-6)

    best_val_acc = 0.0; best_state = None
    patience     = 10; no_improve  = 0   # ← increased from 7

    print(f"\n      Training {NUM_EPOCHS} epochs  "
          f"(train={len(fold_tr)}, val={len(fold_vl)})")

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_acc         = train_epoch(model, tr_loader, optimizer, criterion, scheduler)
        vl_loss, vl_acc, _, __ = eval_epoch(model, vl_loader, criterion)

        flag = ""
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve   = 0; flag = " ← best"
        else:
            no_improve += 1

        print(f"      Ep {epoch:02d}/{NUM_EPOCHS}  "
              f"tr_acc={tr_acc:.4f}  vl_acc={vl_acc:.4f}  "
              f"best={best_val_acc:.4f}{flag}")

        if no_improve >= patience:
            print(f"      ⏹  Early stop at epoch {epoch}")
            break

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    _, _, vl_preds, vl_labels = eval_epoch(model, vl_loader, criterion)
    test_logits = predict(model, test_df, n_tta=8)   # ← 8 TTA passes

    del optimizer, scheduler, tr_loader, vl_loader, best_state
    torch.cuda.empty_cache(); gc.collect()
    return model, vl_preds, vl_labels, test_logits


def train_one_run(train_df, val_df, test_df, dataset_name, run_id, seeds=None):
    if seeds is None:
        seeds = list(range(NUM_SEEDS))

    run_metrics     = []
    all_test_logits = []
    criterion       = HybridLoss()
    full_train      = pd.concat([train_df, val_df]).reset_index(drop=True)

    # ── 5-Fold Cross Validation (architecture diagram) ───────
    skf = StratifiedKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=run_id)

    for fold, (tr_idx, vl_idx) in enumerate(
        skf.split(full_train, full_train["label"]), start=1
    ):
        fold_tr = full_train.iloc[tr_idx].reset_index(drop=True)
        fold_vl = full_train.iloc[vl_idx].reset_index(drop=True)
        print(f"\n  ┌─ Fold {fold}/{NUM_FOLDS}  train={len(fold_tr)}  val={len(fold_vl)} ─┐")

        fold_logits = []
        # ── 3 Seeds per Fold (architecture diagram) ───────────
        for seed in seeds:
            print(f"  │  Seed {seed} ...")
            model, vl_preds, vl_labels, test_logits = _train_one_model(
                fold_tr, fold_vl, test_df, seed, run_id, fold, criterion)

            m = compute_metrics(vl_labels, vl_preds, dataset_name, run_id, fold, seed)
            run_metrics.append(m)
            fold_logits.append(test_logits)
            print(f"  │  ✔ Seed {seed}  val_acc={m['accuracy']:.4f}  f1={m['f1_score']:.4f}")

            del model; torch.cuda.empty_cache(); gc.collect()

        # Average logits across seeds within the fold
        all_test_logits.append(torch.stack(fold_logits).mean(0))
        print(f"  └─ Fold {fold} done ─┘")

    # ── Ensemble Strategy: Average Logits → Softmax ──────────
    ensemble_logits = torch.stack(all_test_logits).mean(0)   # average over 15 models
    ensemble_preds  = ensemble_logits.argmax(dim=1).numpy()

    if (test_df["label"] >= 0).all():
        m_test = compute_metrics(test_df["label"].values, ensemble_preds,
                                 dataset_name, run_id, "ensemble", "all")
        run_metrics.append(m_test)
        save_confusion_matrix(test_df["label"].values, ensemble_preds,
                              dataset_name, run_id, "_ensemble")
        print(f"\n  ✅ Ensemble test → acc={m_test['accuracy']:.4f}  "
              f"f1={m_test['f1_score']:.4f}")

    del criterion, all_test_logits, ensemble_logits
    torch.cuda.empty_cache(); gc.collect()
    return run_metrics

print("✅ CELL 10 complete")


✅ CELL 10 complete


In [15]:
# ============================================================
# CELL 11 — MAIN LOOP (Messidor-2 × 10 Runs)
# All metrics → CSV; averaged metrics → inline bar chart
# ============================================================

def run_dataset(dataset_name):
    print(f"\n{'='*62}")
    print(f"  DATASET : {dataset_name}")
    print(f"  Strategy: {NUM_FOLDS}-Fold CV × {NUM_SEEDS} Seeds = "
          f"{NUM_FOLDS*NUM_SEEDS} models/run × {NUM_RUNS} runs = "
          f"{NUM_FOLDS*NUM_SEEDS*NUM_RUNS} total models")
    print(f"{'='*62}")

    root      = DATASET_PATHS[dataset_name]
    loader_fn = LOADERS[dataset_name]
    train_df, val_df, test_df = loader_fn(root)

    # ── Distribution chart ───────────────────────────────────
    plot_distribution(dataset_name, train_df, val_df, test_df)

    # ── Preprocessing visualisation ──────────────────────────
    sample_path = next(
        (str(r["path"]) for _, r in train_df.iterrows() if os.path.exists(str(r["path"]))), None)
    if sample_path:
        visualise_preprocessing(sample_path, dataset_name)

    all_metrics = []
    for run_id in range(1, NUM_RUNS + 1):
        t0 = time.time()
        print(f"\n  ▶▶  RUN {run_id}/{NUM_RUNS}")
        run_metrics = train_one_run(train_df, val_df, test_df,
                                    dataset_name, run_id,
                                    seeds=list(range(NUM_SEEDS)))
        all_metrics.extend(run_metrics)
        csv_path = append_metrics_csv(run_metrics, dataset_name)
        elapsed  = time.time() - t0
        print(f"  ✔ Run {run_id} complete  ({elapsed/60:.1f} min)  → {csv_path}")

        # Print per-run summary
        run_only = [m for m in run_metrics if m["fold"] == "ensemble"]
        if run_only:
            r = run_only[-1]
            print(f"     acc={r['accuracy']:.4f}  prec={r['precision']:.4f}  "
                  f"sens={r['sensitivity']:.4f}  spec={r['specificity']:.4f}  "
                  f"f1={r['f1_score']:.4f}")

    # ── Averaged metrics → CSV row + inline bar chart ────────
    avg = plot_averaged_metrics_bar(all_metrics, dataset_name)

    if avg["accuracy"] >= 0.98:
        print(f"\n  ✅ 98% TARGET MET — avg accuracy {avg['accuracy']:.4f}")
    else:
        print(f"\n  ℹ️  Avg accuracy {avg['accuracy']:.4f} (target 98%)")
        print(f"     Consider more epochs or dataset augmentation.")

    return avg


# ── Run ───────────────────────────────────────────────────────
summary_records = {}
for ds_name in ["Messidor2"]:
    try:
        avg = run_dataset(ds_name)
        summary_records[ds_name] = avg
    except Exception as exc:
        import traceback; traceback.print_exc()

# ── Final Summary ─────────────────────────────────────────────
print(f"\n{'='*62}")
print(f"  FINAL RESULT — MESSIDOR-2")
print(f"{'='*62}")
if summary_records:
    rows = []
    for ds, avg_s in summary_records.items():
        row = {"Dataset": ds}
        row.update({k.replace("_"," ").title(): f"{v:.4f}" for k, v in avg_s.items()})
        rows.append(row)
    df_sum = pd.DataFrame(rows)
    print(df_sum.to_string(index=False))
    summary_csv = os.path.join(OUTPUT_DIR, "Messidor2_FINAL_SUMMARY.csv")
    df_sum.to_csv(summary_csv, index=False)
    print(f"\n  📁 Final Summary CSV → {summary_csv}")
    print(f"  📁 All Metrics CSV   → {os.path.join(OUTPUT_DIR, 'Messidor2_metrics.csv')}")

print(f"  📁 Results folder : {OUTPUT_DIR}")
print("\n✅ ALL DONE — CELL 11 complete")



  DATASET : Messidor2
  Strategy: 5-Fold CV × 3 Seeds = 15 models/run × 10 runs = 150 total models

  FINAL RESULT — MESSIDOR-2
  📁 Results folder : /home/user2007/Dr_retinopathy/results

✅ ALL DONE — CELL 11 complete


Traceback (most recent call last):
  File "/tmp/ipykernel_1343674/2848303666.py", line 63, in <module>
    avg = run_dataset(ds_name)
  File "/tmp/ipykernel_1343674/2848303666.py", line 15, in run_dataset
    loader_fn = LOADERS[dataset_name]
NameError: name 'LOADERS' is not defined
